In [22]:
# scipy.interpolate.RectBivariateSpline wrapped in PyTorch 
# RectBivariateSpline is efficient and versatile (derivative evaluation ...)
# But it works only in 2D


In [23]:
# Define RectI (Rectangular Interpolation) and other items

import numpy as np
import torch
from scipy.interpolate import RectBivariateSpline
from scipy.sparse.linalg import spsolve

torch.set_default_dtype(torch.float64)
device = 'cuda'

def pt_2d_interpolation(x, y, u, **kwargs):
    """
    Return a PyTorch-compatible 2D interpolator.
    Parameters
    ----------
    x: torch.Tensor, shape (Nx,), x-coordinates of the rectangular grid
    y: torch.Tensor, shape (Ny,), y-coordinates of the rectangular grid
    u: torch.Tensor, shape (Nx, Ny), u[i,j] = f(x[i], y[j])
    Returns
    ----------
    interp: callable
        xq and yq are either torch.Tensors or numpy arrays
        interp(xq, yq) returns a torch.Tensor approximating f(xq, yq).
        interp(xq, yq, dy=1) returns a torch.Tensor approximating df/dy(xq, yq).
    """
    interp_np = RectBivariateSpline( x.cpu().numpy(), y.cpu().numpy(),
        u.cpu().numpy(), **kwargs)
    def interp(xq, yq, **kwargs):
        xq, yq = torch.broadcast_tensors(torch.as_tensor(xq), torch.as_tensor(yq))
        # This is to make it work when xq or yq is a numpy array or a scalar
        z = interp_np.ev(xq.cpu().numpy(), yq.cpu().numpy(), **kwargs)
        return torch.from_numpy(z).to(device=u.device, dtype=u.dtype)
    return interp
    # interp has many useful methods
    #     interp.ev(xq, yq) # set dy = 1 to evaluate df/dy
    #     interp.integral(...)
    #     interp.get_coeffs()
    #     interp.get_knots()
    #     interp.get_residual()
    
def err_of_interp(f, nx, **kwargs):
    ny = nx; dx = 1/nx; dy = 1/ny
    
    # generate the rectangular grid and interpolation data
    x = torch.arange(0, nx+1, device=device)*dx
    y = torch.arange(0, ny+1, device=device)*dy
    x2d, y2d = torch.meshgrid(x, y, indexing='ij')
    u = f(x2d, y2d) # interpolation data

    # construct the interpolation function
    interp_pt = pt_2d_interpolation(x, y, u, **kwargs)

    uq_pred = interp_pt(xq, yq) # interpolation prediction
    uq_true = f(xq, yq) # true value
    
    err_np = (uq_pred-uq_true).abs().cpu().numpy()
    err_rms = np.sqrt((err_np**2).mean())
    return (err_rms, err_np)
